Once you’ve fine-tuned your BERT (or DistilBERT) model with the Hugging Face Trainer, you can reload it anytime for inference — even in a new script, notebook, or Streamlit app — without retraining.

By default, the fine-tuned model and tokenizer are stored in your output_dir (for example ./bert_output).

But you can also save manually:
    trainer.save_model("./distilbert_output")   # saves weights + config
    tokenizer.save_pretrained("./distilbert_output")

This folder will now contain:
    config.json
    pytorch_model.bin
    tokenizer.json
    vocab.txt
    special_tokens_map.json
    training_args.bin

You have a checkpoint directory (checkpoint-500/) created automatically by the Hugging Face Trainer.
Here’s what each file is and how to load your fine-tuned model properly from it.

In [1]:
# Load model

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Path to your checkpoint folder
model_path = "D:/dDev/AI_Language/Learn/distilbert_output/checkpoint-6250"

# Load tokenizer and model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)

# Set model to evaluation mode
model.eval()

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


Note: You can use .from_pretrained(model_path) directly — it will detect model.safetensors automatically.

In [2]:
# Make predictions on new text
import torch

texts = [
    "I love this movie!", 
    "This film was terrible.",
    "This movie is not bad."
]
inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)

labels = ["Negative", "Positive"]
for text, prob in zip(texts, probs):
    pred = labels[prob.argmax()]
    print(f"{text}\n → {pred} ({prob.max().item()*100:.2f}% confidence)\n")


I love this movie!
 → Positive (99.81% confidence)

This film was terrible.
 → Negative (99.84% confidence)

This movie is not bad.
 → Positive (99.05% confidence)



In [3]:
# Make a predication on one sentence:

text = "The movie was surprisingly good, I loved the acting!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)    # turns scores into probabilities
    pred = torch.argmax(probs, dim=1).item()        # chooses the most likely class

    print("Logits:", outputs.logits)
    print("Probabilities:", probs)
    print("Predicted index:", pred)

labels = ["Negative", "Positive"]
pred_prob = probs[0, pred].item()
pred_label = labels[pred]
print(f"Predicted: {pred_label} ({pred_prob:.3f})")

# print("Prediction logits:", outputs.logits)
# print("Predicted class:", labels[pred])


Logits: tensor([[-3.2134,  3.0316]])
Probabilities: tensor([[0.0019, 0.9981]])
Predicted index: 1
Predicted: Positive (0.998)
